# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [28]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully:", bool(HF_TOKEN))

Token loaded successfully: True


In [29]:
!pip -q install -U duckdb

In [30]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Check Colab Secrets."

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection created.")
print("Hugging Face secret configured.")

DuckDB connection created.
Hugging Face secret configured.


In [31]:
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
LIMIT 5
"""

preview = con.execute(query).df()
display(preview)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


### 1. Unit of Analysis and Time Window

**Unit of analysis:** One row represents one reporting date, one pseudonymized client, and one pseudonymized content item in the `fact_content_daily_performance` table.

**Development time window:** March 1–31, 2026, using the `month = '2026-03'` partition.

**Verification results:**
- Total rows: 9,841,378
- Distinct reporting dates: 31
- Distinct clients: 55
- Distinct content items: 331,437
- Distinct grain keys: 9,841,378
- Duplicate grain rows: 0
- Missing reporting dates: 0

The count of distinct grain keys equals the total row count, confirming that no duplicate combinations of reporting date, client hash, and content hash were observed in this slice. The observed reporting dates range from March 1 to March 31, 2026.


In [32]:
# Section 1: Verify the unit of analysis and time window

daily_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

grain_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_report_dates,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_grain_keys,
        COUNT(*) - COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS duplicate_grain_rows
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(grain_check)


month_check = con.execute(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS distinct_report_dates,
        COUNT(*) FILTER (
            WHERE report_date IS NULL
        ) AS null_report_date_rows
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(month_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_report_dates,distinct_clients,distinct_content_items,distinct_grain_keys,duplicate_grain_rows
0,9841378,31,55,331437,9841378,0


,march_rows,first_report_date,last_report_date,distinct_report_dates,null_report_date_rows
0,9841378,2026-03-01,2026-03-31,31,0


### 2. Fields: Feature, Label, Context, and Excluded

**Features:** The proposed five-feature frame uses `prev_gsc_impressions`, `prev_gsc_clicks`, `prev_gsc_avg_position`, `prev_gsc_available`, and `prev_ga4_available`. These fields describe the previous calendar day for the same client and content item. The numeric GSC measurements are used only when the previous day's GSC data is confirmed available. This design reduces the risk of using information from the outcome day.

- `prev_gsc_impressions`: Knowable at the decision moment because it records Google Search Console impressions from the previous calendar day.
- `prev_gsc_clicks`: Knowable at the decision moment because it records Google Search Console clicks from the previous calendar day.
- `prev_gsc_avg_position`: Knowable at the decision moment because it records the previous day's average search position.
- `prev_gsc_available`: Knowable at the decision moment because it indicates whether GSC data was confirmed available for the previous day.
- `prev_ga4_available`: Knowable at the decision moment because it indicates whether GA4 data was confirmed available for the previous day.

**Label:** The binary label is whether a content item receives at least one GSC click on the current reporting date (`gsc_clicks > 0`). It is defined only when `gsc_data_available IS TRUE`. It represents observed same-day click activity; the experiment uses previous-day information to assess whether this outcome can be predicted.

**Context:** `report_date` and `month` describe the reporting period. `client_hash_id` and `content_hash_id` identify the pseudonymized client and content item. The query-level table contains `query_hash_id`, `window_start`, and `window_end`, which describe its query and measurement window.

**Excluded:** Current-day `gsc_clicks`, same-day outcome metrics, and query-window click metrics such as `clicks_90d`, `clicks_last30`, and `clicks_prev30` are excluded from the honest predictor set when they overlap the outcome period. Identifiers are retained for grouping and verification, not treated as meaningful numeric predictors. Private client names, URLs, and query text are not required.

**Missing-data consideration:** Only 3,611,061 of 9,841,378 March rows have confirmed GSC availability. Rows without confirmed GSC availability must not automatically be labelled as zero-click observations.

In [33]:
# Section 2: Verify source availability and missingness

source_quality = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NULL
        ) AS missing_clicks_when_gsc_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions IS NULL
        ) AS missing_impressions_when_gsc_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_sessions IS NULL
        ) AS missing_sessions_when_ga4_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS rows_without_confirmed_gsc_availability

    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(source_quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,missing_clicks_when_gsc_available,missing_impressions_when_gsc_available,missing_sessions_when_ga4_available,rows_without_confirmed_gsc_availability
0,9841378,3611061,0,0,0,6230317


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [34]:
# Section 3: Verification queries
# Query 1 — Verify uniqueness of the declared row grain

q1 = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_grain_keys
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(q1)

# Query 2 — Verify March row count and reporting window

q2 = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(q2)

# Query 3 — Verify source availability and missing values

q3 = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NULL
        ) AS missing_clicks_when_available,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_sessions IS NULL
        ) AS missing_sessions_when_available
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
""").df()

display(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


,row_count,first_date,last_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


,total_rows,gsc_available_rows,missing_clicks_when_available,missing_sessions_when_available
0,9841378,3611061,0,0


In [35]:
# Inspect the query-level table schema using the correct file path

query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

query_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{query_path}')
""").df()

display(query_schema)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [36]:
# Inspect the query-level data without exposing private query text

query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

query_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT query_hash_id) AS distinct_query_hashes,
        MIN(window_start) AS earliest_window_start,
        MAX(window_end) AS latest_window_end,
        COUNT(*) FILTER (
            WHERE window_start IS NULL OR window_end IS NULL
        ) AS rows_with_missing_window_bounds
    FROM read_parquet('{query_path}')
""").df()

display(query_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_clients,distinct_content_items,distinct_query_hashes,earliest_window_start,latest_window_end,rows_with_missing_window_bounds
0,2414248,52,133852,1180090,2026-04-02,2026-06-30,0


In [37]:

# Section 2: Build a five-feature frame using the previous calendar day.

feature_frame_query = f"""
WITH lagged AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_clicks,

        LAG(report_date) OVER w AS prev_report_date,
        LAG(gsc_data_available) OVER w AS prev_gsc_available,
        LAG(ga4_data_available) OVER w AS prev_ga4_available,
        LAG(gsc_impressions) OVER w AS raw_prev_gsc_impressions,
        LAG(gsc_clicks) OVER w AS raw_prev_gsc_clicks,
        LAG(gsc_avg_position) OVER w AS raw_prev_gsc_avg_position

    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'

    WINDOW w AS (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
    )
)
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    CASE
        WHEN prev_gsc_available IS TRUE
        THEN raw_prev_gsc_impressions
        ELSE NULL
    END AS prev_gsc_impressions,

    CASE
        WHEN prev_gsc_available IS TRUE
        THEN raw_prev_gsc_clicks
        ELSE NULL
    END AS prev_gsc_clicks,

    CASE
        WHEN prev_gsc_available IS TRUE
        THEN raw_prev_gsc_avg_position
        ELSE NULL
    END AS prev_gsc_avg_position,

    CAST(prev_gsc_available IS TRUE AS INTEGER)
        AS prev_gsc_available,

    CAST(prev_ga4_available IS TRUE AS INTEGER)
        AS prev_ga4_available,

    CAST(gsc_clicks > 0 AS INTEGER) AS label_clicks_positive,

    -- Deliberately retained only for the leakage demonstration.
    gsc_clicks AS leak_current_day_clicks

FROM lagged
WHERE report_date = prev_report_date + INTERVAL 1 DAY
  AND gsc_data_available IS TRUE
"""

feature_df = con.execute(feature_frame_query).df()

print("Feature-frame rows:", len(feature_df))
print("Feature-frame columns:", feature_df.shape[1])
print("\nLabel distribution:")
display(
    feature_df["label_clicks_positive"]
    .value_counts()
    .sort_index()
    .rename("row_count")
    .to_frame()
)
display(feature_df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 3506793
Feature-frame columns: 10

Label distribution:


,row_count
label_clicks_positive,
0,3101440
1,405353


,report_date,client_hash_id,content_hash_id,prev_gsc_impressions,prev_gsc_clicks,prev_gsc_avg_position,prev_gsc_available,prev_ga4_available,label_clicks_positive,leak_current_day_clicks
0,2026-03-04,client_08a6a72ff48e62c0,content_0277b30d14992551,<NA>,<NA>,NaN,0,0,0,0
1,2026-03-06,client_08a6a72ff48e62c0,content_0277b30d14992551,<NA>,<NA>,NaN,0,0,0,0
2,2026-03-07,client_08a6a72ff48e62c0,content_0277b30d14992551,1,0,11.0,1,0,0,0
3,2026-03-02,client_08a6a72ff48e62c0,content_02a8bf2c0b7ee8c4,1,0,83.0,1,0,0,0
4,2026-03-04,client_08a6a72ff48e62c0,content_02a8bf2c0b7ee8c4,<NA>,<NA>,NaN,0,0,0,0


In [38]:
# Check the candidate label distribution on GSC-available rows

label_check = con.execute(f"""
    SELECT
        COUNT(*) AS eligible_rows,
        COUNT(*) FILTER (WHERE gsc_clicks > 0) AS rows_with_clicks,
        COUNT(*) FILTER (WHERE gsc_clicks = 0) AS rows_with_zero_clicks,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS rows_with_missing_clicks
    FROM read_parquet('{daily_path}', hive_partitioning = true)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

display(label_check)

,eligible_rows,rows_with_clicks,rows_with_zero_clicks,rows_with_missing_clicks
0,3611061,417981,3193080,0


In [39]:

# Leakage experiment: compare honest predictors against an invalid predictor.
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

features = [
    "prev_gsc_impressions",
    "prev_gsc_clicks",
    "prev_gsc_avg_position",
    "prev_gsc_available",
    "prev_ga4_available",
]

train_df, test_df = train_test_split(
    feature_df,
    test_size=0.20,
    random_state=42,
    stratify=feature_df["label_clicks_positive"],
)

y_train = train_df["label_clicks_positive"]
y_test = test_df["label_clicks_positive"]

# Honest model: only information from the previous day.
honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

honest_model.fit(train_df[features], y_train)
honest_pred = honest_model.predict(test_df[features])

# Invalid model: current-day clicks directly reveal the target label.
leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(
    train_df[["leak_current_day_clicks"]],
    y_train,
)
leaky_pred = leaky_model.predict(
    test_df[["leak_current_day_clicks"]]
)

comparison = pd.DataFrame({
    "model": ["Honest: previous-day features", "Leaky: current-day clicks"],
    "accuracy": [
        accuracy_score(y_test, honest_pred),
        accuracy_score(y_test, leaky_pred),
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, honest_pred),
        balanced_accuracy_score(y_test, leaky_pred),
    ],
})

display(comparison)


,model,accuracy,balanced_accuracy
0,Honest: previous-day features,0.848049,0.777444
1,Leaky: current-day clicks,1.000000,1.000000


### 4. Data limits

This slice has several limitations:

1. **Uneven source availability:** Only 3,611,061 of 9,841,378 March rows have confirmed GSC availability. The eligible rows are not necessarily representative of every client or content item.
2. **Imbalanced label:** Among rows with confirmed GSC availability, 417,981 have at least one click and 3,193,080 have zero clicks. A model can achieve misleadingly high accuracy by favouring the majority class, so balanced accuracy should also be reported.
3. **Limited time window:** The development slice covers March 1–31, 2026. One month cannot establish seasonal patterns or reliable long-term generalisation.
4. **Different measurement windows:** The query-level table's observed windows fall after the March development period. These query-level metrics cannot be treated as information available for a March prediction. Their suitability for a later prediction task would require a separate temporal review.
5. **Observational data:** The data measures recorded search and analytics activity. It cannot, by itself, prove that a content change caused a change in clicks or sessions.
6. **Leakage risk:** A feature derived from current-day clicks reveals the label. The deliberately leaky model is included only as a demonstration and must not be used to report predictive performance.

The resulting model is suitable only for exploratory, directional decision support. Its performance should be evaluated on temporally separate data before making claims about future performance.

In [40]:

# Section 4: Data-limit checks based on observed March results.

total_march_rows = 9841378
gsc_available_rows = 3611061
positive_rows = 417981
zero_click_rows = 3193080

limits_check = pd.DataFrame({
    "measure": [
        "March rows with confirmed GSC availability",
        "Eligible rows with at least one click",
        "Eligible rows with zero clicks",
    ],
    "count": [
        gsc_available_rows,
        positive_rows,
        zero_click_rows,
    ],
    "percentage": [
        100 * gsc_available_rows / total_march_rows,
        100 * positive_rows / (positive_rows + zero_click_rows),
        100 * zero_click_rows / (positive_rows + zero_click_rows),
    ],
})

limits_check["percentage"] = limits_check["percentage"].round(2)
display(limits_check)


,measure,count,percentage
0,March rows with confirmed GSC availability,3611061,36.69
1,Eligible rows with at least one click,417981,11.58
2,Eligible rows with zero clicks,3193080,88.42


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.